# Watermark Segmentation - Colab Training

Trains a single-class YOLO11-seg model to segment the **AriaTender** watermark.

**Upload one file:** `wm_dataset_colab.zip` (37 MB - 236 images + 236 YOLO-seg labels).

**Bring back:** `best.pt`.

---
### Why the settings differ from the local run

Locally this ran on an MX150 (~2 GB VRAM) at ~72 s/iteration - about 60 hours for 60 epochs,
while spilling past VRAM into shared memory. A Colab T4/L4 removes that constraint, so this
notebook raises `imgsz` back to **1024**. That matters specifically here: the watermark contains
a thin-ring shield logo whose strokes get destroyed at low resolution, and that is the exact
shape the current SAM-based detector already fails on. Resolution is the main lever for it.

**Training data is entirely synthetic** (the watermark composited onto generated Persian
document pages). Whether it transfers to real documents is the open question - judge that on
real images after bringing `best.pt` back, not from the numbers in this notebook.


## 1. Check the GPU

If this fails: `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('device:', torch.cuda.get_device_name(0))


## 2. Install ultralytics

Pinned to the version used locally so results stay comparable.


In [ ]:
%pip -q install ultralytics==8.4.142
import ultralytics; ultralytics.checks()


## 3. Upload the dataset

Run the cell, then choose **`wm_dataset_colab.zip`**.

If the browser upload is slow or drops, use the Google Drive cell below instead.


In [ ]:
import pathlib
from google.colab import files

if not pathlib.Path('wm_dataset_colab.zip').exists():
    uploaded = files.upload()   # choose wm_dataset_colab.zip
else:
    print('wm_dataset_colab.zip already present, skipping upload.')


*Alternative - Google Drive.* Skip if the upload above worked. Put the zip in your Drive root first.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/wm_dataset_colab.zip' /content/


## 4. Unzip and write `data.yaml` with ABSOLUTE paths

This is deliberate, not boilerplate. Ultralytics resolves a relative `path:` against its own
`settings['datasets_dir']` - **not** the working directory - so a relative path silently looks in
the wrong place and training reads nothing. Absolute paths avoid that, and the cell then confirms
it using ultralytics' own resolver rather than assuming.


In [ ]:
import zipfile, pathlib, yaml

ROOT = pathlib.Path('/content/wm_dataset_out')
if not ROOT.exists():
    with zipfile.ZipFile('wm_dataset_colab.zip') as z:
        z.extractall('/content')

data = {
    'path':  str(ROOT),          # ABSOLUTE - see note above
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['watermark'],
}
yaml_path = pathlib.Path('/content/data.yaml')
yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print(yaml_path.read_text())

from ultralytics.data.utils import check_det_dataset
resolved = check_det_dataset(str(yaml_path))
print('resolved train ->', resolved['train'])
print('resolved val   ->', resolved['val'])


## 5. Sanity-check the dataset before spending GPU time

Confirms every image has a matching label, and counts the intentional **negatives** (empty label
files = clean pages with no watermark). Ultralytics treats those as background images, which is
wanted - they teach the model not to fire on clean documents. A negative count of 0 here means
the upload is wrong.


In [ ]:
for split in ['train', 'val']:
    imgs = sorted((ROOT/'images'/split).glob('*'))
    labs = sorted((ROOT/'labels'/split).glob('*.txt'))
    stems_i = {p.stem for p in imgs}
    stems_l = {p.stem for p in labs}
    empties = [p for p in labs if p.stat().st_size == 0]
    print(f'{split:5} images={len(imgs):4d} labels={len(labs):4d} '
          f'unmatched={len(stems_i ^ stems_l):3d} negatives={len(empties):3d}')
    assert not (stems_i ^ stems_l), f'{split}: image/label filenames do not match'


## 6. Train

201 train / 35 val images. On a T4 expect roughly 20-40 minutes.

`patience=30` early-stops if validation stops improving - the dataset is small and will overfit
quickly. On CUDA OOM: drop `batch` to 8, then 4, and only reduce `imgsz` as a last resort.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')   # COCO-pretrained, nano

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=1024,      # thin-ring shield logo needs the resolution
    batch=16,
    patience=30,
    seed=0,
    project='/content/runs',
    name='wm_seg',
    plots=True,
)


## 7. Validate and look at predictions

These mAP numbers are on **held-out synthetic** data. They show the model learned the training
distribution; they do **not** show it works on real scans. The plotted validation batches below
are the more informative output.


In [ ]:
metrics = model.val()
print('mask mAP50    :', round(float(metrics.seg.map50), 4))
print('mask mAP50-95 :', round(float(metrics.seg.map), 4))
print('mask precision:', round(float(metrics.seg.mp), 4))
print('mask recall   :', round(float(metrics.seg.mr), 4))


In [ ]:
import glob
from IPython.display import Image as IPyImage, display

for p in sorted(glob.glob('/content/runs/wm_seg/val_batch*_pred.jpg'))[:3]:
    print(p)
    display(IPyImage(filename=p, width=900))


## 8. Download `best.pt`

This is the file to send back.


In [ ]:
import pathlib
from google.colab import files

best = pathlib.Path('/content/runs/wm_seg/weights/best.pt')
print('exists:', best.exists(), '| size:', round(best.stat().st_size / 1e6, 2), 'MB')
files.download(str(best))


*Optional - also copy the whole run to Drive, in case the session disconnects.*


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/runs/wm_seg '/content/drive/MyDrive/wm_seg_run'
